# 📓 Notebook 01 — LLM Basics
---
**Phase 1 · LLM Fundamentals**

> Master the core building blocks: API calls, prompt engineering, and model parameters.
> Every concept here is interview-critical and builds the foundation for everything ahead.

### What You'll Learn
| Topic | Why It Matters |
|-------|---------------|
| Multi-model setup | Work with OpenAI, Gemini, Claude from one codebase |
| Tokenization | Understand cost, context limits, and truncation |
| Temperature & sampling | Control creativity vs determinism |
| Prompt engineering | Zero-shot, few-shot, CoT, self-consistency |
| System prompts | Shape model behavior reliably |

### 🏗️ Build: Multi-Model Q&A Chatbot

# --- Setup: LLM Configuration ---
# All config is centralized in setup_llm.py
# Proxy mode auto-activates when LITELLM_PROXY_URL is set in .env

import os, sys
import litellm

from setup_llm import DEFAULT_MODEL, DEFAULT_EMBEDDING, is_proxy_mode

mode = "🔗 proxy" if is_proxy_mode() else "🔑 direct"
print(f"✅ Mode: {mode}")
print(f"🤖 Model: {DEFAULT_MODEL}")

In [ ]:
# --- Install dependencies (run once) ---
# !pip install litellm openai google-genai anthropic python-dotenv tiktoken

import os
from dotenv import load_dotenv

# Load API keys from .env file
load_dotenv()

# Verify at least one key is set
providers = {
    "OpenAI": os.getenv("OPENAI_API_KEY"),
    "Google": os.getenv("GOOGLE_API_KEY"),
    "Anthropic": os.getenv("ANTHROPIC_API_KEY"),
}
available = [k for k, v in providers.items() if v and not v.startswith("your-")]
print(f"✅ Available providers: {', '.join(available) if available else '⚠️ No API keys found! Edit .env file.'}")

# ask_llm — Universal helper function for all notebooks

def ask_llm(prompt, model=None, temperature=0.7, max_tokens=500, system_prompt=None):
    """Universal LLM call — works with any provider via litellm."""
    messages = []
    if system_prompt:
        messages.append({"role": "system", "content": system_prompt})
    messages.append({"role": "user", "content": prompt})
    
    response = litellm.completion(
        model=model or DEFAULT_MODEL,
        messages=messages,
        temperature=temperature,
        max_tokens=max_tokens,
    )
    return response.choices[0].message.content

# Test it!
result = ask_llm("What is the capital of France? Answer in one sentence.")
print(f"\n📝 Response: {result}")

In [ ]:
import litellm

# Set default model from .env, or fall back
DEFAULT_MODEL = os.getenv("DEFAULT_MODEL", "gpt-4o-mini")
print(f"🤖 Using model: {DEFAULT_MODEL}")

def ask_llm(prompt, model=None, temperature=0.7, max_tokens=500, system_prompt=None):
    """Universal LLM call — works with any provider via litellm."""
    messages = []
    if system_prompt:
        messages.append({"role": "system", "content": system_prompt})
    messages.append({"role": "user", "content": prompt})
    
    response = litellm.completion(
        model=model or DEFAULT_MODEL,
        messages=messages,
        temperature=temperature,
        max_tokens=max_tokens,
    )
    return response.choices[0].message.content

# Test it!
result = ask_llm("What is the capital of France? Answer in one sentence.")
print(f"\n📝 Response: {result}")

---
## 2. Understanding Tokenization

> **Tokens** are the atomic units LLMs work with — not characters, not words.

### Why This Matters
- **Cost** is calculated per token (input + output)
- **Context window** is measured in tokens (e.g., GPT-4o = 128K tokens)
- **Truncation** happens at token boundaries, not word boundaries

### Token Facts (Interview-Critical)
| Fact | Detail |
|------|--------|
| 1 token ≈ | 4 characters in English, ¾ of a word |
| "ChatGPT" = | 3 tokens: ["Chat", "G", "PT"] |
| 100 tokens ≈ | 75 English words |
| Code is expensive | Variable names and syntax cost more tokens |
| Non-English | Languages like Hindi/Chinese use more tokens per word |

In [ ]:
import tiktoken

# OpenAI's tokenizer (works offline, great for understanding)
encoder = tiktoken.encoding_for_model("gpt-4o")

examples = [
    "Hello, world!",
    "Artificial Intelligence is transforming healthcare.",
    "def fibonacci(n): return n if n <= 1 else fibonacci(n-1) + fibonacci(n-2)",
    "The quick brown fox jumps over the lazy dog.",
    "नमस्ते दुनिया",  # Hindi: "Hello World"
]

print("📊 Tokenization Analysis")
print("=" * 70)
for text in examples:
    tokens = encoder.encode(text)
    print(f"\nText: \"{text}\"")
    print(f"  Tokens ({len(tokens)}): {tokens}")
    print(f"  Decoded: {[encoder.decode([t]) for t in tokens]}")
    print(f"  Chars/token ratio: {len(text)/len(tokens):.1f}")

### 💡 Token Economics

```
Cost = (input_tokens × input_price) + (output_tokens × output_price)
```

| Model | Input (per 1M tokens) | Output (per 1M tokens) |
|-------|----------------------|----------------------|
| GPT-4o-mini | $0.15 | $0.60 |
| GPT-4o | $2.50 | $10.00 |
| Gemini Flash | $0.075 | $0.30 |
| Claude 3 Haiku | $0.25 | $1.25 |

> **Interview Tip:** *"How would you reduce LLM costs?"*  
> Answer: Use cheaper models for simple tasks (routing), cache frequent queries, minimize prompt length, use batch APIs.

---
## 3. Temperature & Sampling Parameters

These control **how the model selects the next token**.

### The Core Parameters

| Parameter | Range | Effect |
|-----------|-------|--------|
| `temperature` | 0.0 – 2.0 | Controls randomness. 0 = deterministic, 1 = balanced, 2 = very random |
| `top_p` | 0.0 – 1.0 | Nucleus sampling. Only considers tokens within top P probability mass |
| `max_tokens` | 1 – model max | Maximum output length |
| `frequency_penalty` | -2.0 – 2.0 | Penalizes repeated tokens |
| `presence_penalty` | -2.0 – 2.0 | Penalizes tokens that already appeared |

> **Key Insight:** Temperature and top_p both control randomness but work differently.  
> Temperature scales the logits; top_p truncates the probability distribution.  
> **Rule:** Use one or the other, not both.

### 🎨 Visualizing Temperature

![How Temperature Controls Randomness](images/temperature_visualization.png)

#### 💡 Intuition for Juniors: Temperature as a "Creativity Dial"

Think of temperature like a dial on a radio:
- **Turn it down (0.0)** → The model always picks the most likely word. Boring but reliable.
- **Turn it up (1.5+)** → The model picks from many possible words, even unlikely ones. Creative but unpredictable.

#### 📐 The Math Behind It (Softmax with Temperature)

Before the model picks a word, it calculates a "score" (logit) for every possible next word.
Temperature modifies how these scores turn into probabilities:

```
probability(word) = exp(score / temperature) / sum(exp(all_scores / temperature))
```

**Worked Example:** Say the model scores 3 words:

| Word | Raw Score (logit) |
|------|------------------|
| "cat" | 5.0 |
| "dog" | 3.0 |
| "fish" | 1.0 |

**At Temperature = 1.0** (normal):
```
exp(5/1) = 148.4,  exp(3/1) = 20.1,  exp(1/1) = 2.7
Total = 171.2
P(cat) = 148.4 / 171.2 = 86.7%  ← Most likely
P(dog) = 20.1  / 171.2 = 11.7%
P(fish) = 2.7  / 171.2 =  1.6%
```

**At Temperature = 0.1** (very cold → deterministic):
```
exp(5/0.1) = exp(50) ≈ 5.2×10²¹  ← HUGE
exp(3/0.1) = exp(30) ≈ 1.1×10¹³
exp(1/0.1) = exp(10) ≈ 2.2×10⁴
P(cat) ≈ 99.999...%  ← Always picks "cat"
```

**At Temperature = 2.0** (hot → creative):
```
exp(5/2) = 12.2,  exp(3/2) = 4.5,  exp(1/2) = 1.6
Total = 18.3
P(cat)  = 12.2 / 18.3 = 66.7%
P(dog)  = 4.5  / 18.3 = 24.6%  ← Much more likely now!
P(fish) = 1.6  / 18.3 =  8.7%
```

> **Key takeaway:** Low temperature → winner takes all. High temperature → everyone gets a chance.

### Softmax from Scratch -- Why It Works This Way

#### Step 0: The Problem
The LLM calculates a raw score (called a **logit**) for every possible next word.
These scores can be any number -- they are NOT probabilities yet:

```
"cat" -> 5.0    "dog" -> 3.0    "fish" -> 1.0    "car" -> -2.0
```

We need to turn these into probabilities (numbers between 0 and 1 that add up to 1).

---

#### Why Not Just Divide by the Total?

Your first instinct might be: divide each score by the sum of all scores.

```
Total = 5 + 3 + 1 + (-2) = 7
P(cat) = 5/7 = 71%    <-- Seems okay...
P(car) = -2/7 = -29%  <-- NEGATIVE probability! Impossible!
```

**Problem:** Negative scores give negative probabilities. That makes no sense!

---

#### Why Not Use Absolute Values?

```
Total = |5| + |3| + |1| + |-2| = 11
P(cat) = 5/11 = 45%
P(car) = 2/11 = 18%   <-- But car had a NEGATIVE score! Should not be 18%!
```

**Problem:** A word the model hated (score = -2) gets an 18% chance. Wrong!

---

#### The Solution: Exponentials (e^x)

The **exponential function** `e^x` has two magical properties:
1. **Always positive** -- `e^x > 0` for any x (even negative x). No negative probabilities!
2. **Amplifies differences** -- small score differences become big probability differences.

```
e^5.0 = 148.4    <-- Big score = big number
e^3.0 = 20.1     <-- Medium score = medium number
e^1.0 = 2.7      <-- Small score = small number
e^(-2) = 0.14    <-- Negative score = tiny number (but still positive!)
```

Now divide each by the total:
```
Total = 148.4 + 20.1 + 2.7 + 0.14 = 171.3

P(cat)  = 148.4 / 171.3 = 86.6%   High score = high probability
P(dog)  = 20.1  / 171.3 = 11.7%   Medium
P(fish) = 2.7   / 171.3 =  1.6%   Low
P(car)  = 0.14  / 171.3 =  0.1%   Negative score = nearly zero (but not negative!)
```

**This is softmax!** `P(word) = e^score / sum(e^all_scores)`

---

#### Where Does Temperature Fit In?

Temperature divides the scores BEFORE applying softmax:

```
P(word) = e^(score / T) / sum(e^(all_scores / T))
```

**Why?** Dividing by small T (like 0.1) makes scores 10x bigger. Exponentials amplify this. Winner takes all.
Dividing by big T (like 2.0) makes scores smaller. Exponentials compress this. Everyone gets a chance.

| Temperature | Effect on Scores | Result |
|------------|-----------------|--------|
| T = 0.1 | 5/0.1 = **50**, 3/0.1 = **30** | Huge gap, cat gets ~100% |
| T = 1.0 | 5/1 = 5, 3/1 = 3 | Normal gap, cat gets ~87% |
| T = 2.0 | 5/2 = 2.5, 3/2 = 1.5 | Small gap, cat gets ~50% |

> **TL;DR:** Softmax uses e^x because it makes all values positive. Temperature controls how peaked the distribution is.

In [ ]:
from IPython.display import HTMLHTML("""<div id="temp-demo" style="font-family:'Segoe UI',system-ui,sans-serif;background:linear-gradient(135deg,#0f0c29,#1a1a2e);padding:24px;border-radius:16px;color:#e0e0e0;max-width:650px;box-shadow:0 8px 32px rgba(0,0,0,0.4);">  <h3 style="margin:0 0 4px;color:#00d2ff;font-size:18px;">&#x1F321; Interactive Temperature Demo</h3>  <p style="margin:0 0 14px;font-size:13px;color:#888;">Drag the slider to see how temperature changes token probabilities</p>  <div style="display:flex;align-items:center;gap:12px;margin-bottom:16px;">    <span style="font-size:12px;color:#6ec6ff;">&#x2744; Deterministic</span>    <input type="range" id="tempSlider" min="1" max="30" value="10" style="flex:1;accent-color:#00d2ff;height:6px;">    <span style="font-size:12px;color:#ff6b6b;">&#x1F525; Creative</span>  </div>  <div style="text-align:center;margin-bottom:14px;">    <span style="font-size:28px;font-weight:bold;color:#00d2ff;" id="tempVal">1.0</span>    <span style="font-size:14px;color:#888;"> temperature</span>  </div>  <div id="bars" style="display:flex;flex-direction:column;gap:8px;"></div>  <div style="margin-top:14px;padding:10px;background:rgba(0,210,255,0.08);border-radius:8px;border-left:3px solid #00d2ff;">    <p id="insight" style="margin:0;font-size:13px;color:#aaa;"></p>  </div></div><script>(function(){  var tokens = [{n:"cat",s:5,c:"#00d2ff"},{n:"dog",s:3,c:"#6ec6ff"},{n:"fish",s:1.5,c:"#a78bfa"},{n:"car",s:0.5,c:"#f472b6"},{n:"tree",s:-0.5,c:"#fb923c"}];  var slider = document.getElementById('tempSlider');  var tempVal = document.getElementById('tempVal');  var barsDiv = document.getElementById('bars');  var insightEl = document.getElementById('insight');  function softmax(scores, T) {    var maxS = Math.max.apply(null, scores);    var exps = scores.map(function(s){return Math.exp((s - maxS) / T);});    var sum = exps.reduce(function(a, b){return a + b;}, 0);    return exps.map(function(e){return e / sum;});  }  function update() {    var T = parseInt(slider.value) / 10;    tempVal.textContent = T.toFixed(1);    var probs = softmax(tokens.map(function(t){return t.s;}), T);    var maxProb = Math.max.apply(null, probs);    var html = '';    for (var i = 0; i < tokens.length; i++) {      var pct = (probs[i] * 100).toFixed(1);      var w = Math.max(probs[i] / maxProb * 100, 2);      html += '<div style="display:flex;align-items:center;gap:8px;">'        + '<div style="width:45px;text-align:right;font-size:13px;font-weight:600;color:' + tokens[i].c + ';">' + tokens[i].n + '</div>'        + '<div style="flex:1;background:rgba(255,255,255,0.06);border-radius:6px;height:28px;overflow:hidden;">'        + '<div style="height:100%;width:' + w + '%;background:linear-gradient(90deg,' + tokens[i].c + ',' + tokens[i].c + '88);border-radius:6px;transition:width 0.3s ease;display:flex;align-items:center;justify-content:flex-end;padding-right:8px;">'        + '<span style="font-size:12px;font-weight:700;color:#fff;text-shadow:0 1px 3px rgba(0,0,0,0.5);">' + pct + '%</span></div></div></div>';    }    barsDiv.innerHTML = html;    if (T <= 0.3) insightEl.innerHTML = '<b>Near-zero temperature</b> -- the model almost ALWAYS picks the highest-scored token. Great for factual answers, code generation.';    else if (T <= 0.8) insightEl.innerHTML = '<b>Low temperature</b> -- top token still wins most of the time, but others have a small chance.';    else if (T <= 1.2) insightEl.innerHTML = '<b>Balanced temperature</b> -- probabilities spread out. Lower-ranked tokens get real chances.';    else insightEl.innerHTML = '<b>High temperature</b> -- probabilities nearly flatten. Even unlikely tokens get picked. Good for brainstorming.';  }  slider.addEventListener('input', update);  update();})();</script>""")

In [ ]:
# Experiment: Temperature's effect on output

prompt = "Write a one-sentence description of artificial intelligence."

print("🌡️ Temperature Experiment")
print("=" * 60)

for temp in [0.0, 0.5, 1.0, 1.5]:
    print(f"\n--- Temperature: {temp} ---")
    for i in range(3):
        result = ask_llm(prompt, temperature=temp, max_tokens=60)
        print(f"  Run {i+1}: {result.strip()}")

In [ ]:
# Practical Guide: When to use which temperature

scenarios = {
    "Code generation": 0.0,
    "Data extraction / JSON": 0.0,
    "Customer support": 0.3,
    "General Q&A": 0.5,
    "Creative writing": 0.8,
    "Brainstorming": 1.2,
    "Poetry / humor": 1.5,
}

print("🎯 Temperature Cheat Sheet")
print("=" * 50)
for task, temp in scenarios.items():
    bar = "█" * int(temp * 10)
    print(f"  {task:<25} temp={temp:<4} {bar}")

---
## 4. System Prompts & Message Roles

### The Message Architecture

Every LLM conversation uses a **message array** with three roles:

```python
messages = [
    {"role": "system",    "content": "You are a helpful assistant."},   # Sets behavior
    {"role": "user",      "content": "What is RAG?"},                   # User input
    {"role": "assistant", "content": "RAG stands for..."},              # Model response
]
```

| Role | Purpose | When Set |
|------|---------|----------|
| `system` | Defines persona, rules, output format | Once, at the start |
| `user` | User's question or instruction | Each turn |
| `assistant` | Model's previous responses (for context) | Multi-turn conversations |

> **Interview Tip:** *"What's the difference between system prompt and user prompt?"*  
> System prompts set persistent behavioral constraints. User prompts provide the task.
> System prompts have higher "authority" — but aren't immune to prompt injection.

In [ ]:
# Demonstrate system prompts shaping behavior

system_prompts = {
    "Pirate": "You are a pirate. Respond to everything in pirate speak. Keep answers under 2 sentences.",
    "Data Scientist": "You are a senior data scientist. Give precise, technical answers with metrics when possible.",
    "5-year-old": "Explain everything as if talking to a 5 year old. Use simple words and analogies.",
}

user_question = "What is machine learning?"

print("🎭 System Prompt Experiment")
print("=" * 60)
for persona, system in system_prompts.items():
    result = ask_llm(user_question, system_prompt=system, temperature=0.5)
    print(f"\n👤 [{persona}]")
    print(f"   {result.strip()}")

---
## 5. Prompt Engineering — The Core Techniques

### Technique Overview

| Technique | Description | Best For |
|-----------|-------------|----------|
| **Zero-shot** | Just ask, no examples | Simple factual questions |
| **Few-shot** | Provide examples of input→output | Classification, formatting |
| **Chain-of-Thought (CoT)** | Ask model to reason step by step | Math, logic, complex reasoning |
| **Self-Consistency** | Generate multiple CoT paths, take majority vote | High-stakes decisions |
| **Role Prompting** | Assign expertise role | Domain-specific tasks |

> **Interview Critical:** Understanding *when* to use each technique is more important than knowing the names.

### 🗺️ Choosing the Right Technique

![Which Prompting Technique Should I Use?](images/prompt_techniques_flowchart.png)

> **Quick decision guide:**
> - Start with **zero-shot** — it's the simplest and often good enough.
> - If the output format is wrong, switch to **few-shot** (show examples).
> - If reasoning is wrong, add **"Think step by step"** (Chain-of-Thought).
> - If accuracy is critical, use **self-consistency** (multiple attempts + vote).

In [ ]:
# ===== ZERO-SHOT =====
print("1️⃣ ZERO-SHOT")
print("-" * 40)
result = ask_llm(
    "Classify this review as POSITIVE, NEGATIVE, or NEUTRAL: 'The food was okay but the service was terrible.'",
    temperature=0
)
print(f"Result: {result}")

In [ ]:
# ===== FEW-SHOT =====
print("\n2️⃣ FEW-SHOT")
print("-" * 40)
few_shot_prompt = """Classify the sentiment of each review.

Review: "Absolutely loved the pasta! Best Italian in town."
Sentiment: POSITIVE

Review: "Waited 45 minutes for cold food. Never again." 
Sentiment: NEGATIVE

Review: "It was fine. Nothing special but nothing wrong either."
Sentiment: NEUTRAL

Review: "The ambiance was great but portions were tiny for the price."
Sentiment:"""

result = ask_llm(few_shot_prompt, temperature=0)
print(f"Result: {result}")

In [ ]:
# ===== CHAIN-OF-THOUGHT (CoT) =====
print("\n3️⃣ CHAIN-OF-THOUGHT")
print("-" * 40)

# WITHOUT CoT
result_no_cot = ask_llm(
    "If a shirt costs $25 and is on 20% sale, and you also have a $5 coupon, how much do you pay?",
    temperature=0
)
print(f"Without CoT: {result_no_cot.strip()}")

# WITH CoT
result_cot = ask_llm(
    "If a shirt costs $25 and is on 20% sale, and you also have a $5 coupon, how much do you pay? Think step by step.",
    temperature=0
)
print(f"\nWith CoT: {result_cot.strip()}")

In [ ]:
# ===== SELF-CONSISTENCY =====
print("\n4️⃣ SELF-CONSISTENCY")
print("-" * 40)

question = "A farmer has 15 sheep. All but 8 die. How many are left?"

answers = []
for i in range(5):
    result = ask_llm(
        f"{question} Think step by step. Give ONLY the final number.",
        temperature=0.7,
        max_tokens=100
    )
    answers.append(result.strip())
    print(f"  Run {i+1}: {result.strip()}")

# Majority vote
from collections import Counter
most_common = Counter(answers).most_common(1)[0]
print(f"\n🏆 Majority answer: {most_common[0]} (appeared {most_common[1]}/5 times)")

### 🧪 Advanced Prompt Patterns

| Pattern | Template | Use Case |
|---------|----------|----------|
| **Instruction + Constraint** | "Do X. Do NOT do Y." | Controlled output |
| **Output formatting** | "Respond in JSON with keys: ..." | Structured data |
| **Persona + Task** | "As a {role}, analyze {input}" | Domain expertise |
| **Step-by-step + Verify** | "Solve step by step, then verify your answer" | Math/Logic |
| **Delimiter separation** | Use ``` or --- to separate sections | Complex prompts |

In [ ]:
# Advanced pattern: Instruction + Constraint + Format
advanced_prompt = """You are a financial analyst. Analyze the following company description and provide:
1. Industry sector
2. Key risk factors (max 3)
3. Growth potential (LOW/MEDIUM/HIGH)

Respond in this exact JSON format:
{
  "sector": "...",
  "risks": ["...", "...", "..."],
  "growth_potential": "..."
}

Company: "TechVault is a 2-year-old startup building encrypted cloud storage 
for healthcare providers. They have 50 enterprise clients and $2M ARR, growing 
at 15% month-over-month. They recently received SOC2 certification."
"""

result = ask_llm(advanced_prompt, temperature=0)
print("📊 Structured Analysis:")
print(result)

---
## 📝 Interview Quiz — LLM Fundamentals

Test your knowledge! Try to answer each question before revealing the answer.

---

### Q1: What is a token? How does tokenization affect LLM cost and behavior?

<details>
<summary>💡 Click to reveal answer</summary>

A **token** is the smallest unit of text an LLM processes — subword pieces, not whole words. 
- "ChatGPT" = 3 tokens: `["Chat", "G", "PT"]`
- Cost is calculated as: `(input_tokens × price) + (output_tokens × price)`
- Context window limits are in tokens, not characters
- Non-English text and code typically use more tokens per word
- Tokenization varies by model — GPT uses BPE (Byte-Pair Encoding), others may differ

</details>

### Q2: What's the difference between `temperature` and `top_p`? Can you use both?

<details>
<summary>💡 Click to reveal answer</summary>

Both control randomness but work differently:
- **Temperature** scales the logit values before softmax. Higher = flatter distribution = more random
- **top_p** (nucleus sampling) truncates the probability distribution, only considering tokens whose cumulative probability ≤ p
- **Best practice:** Use one or the other, not both simultaneously
- `temperature=0` → deterministic (greedy decoding)
- For factual tasks: temp=0, for creative: temp=0.7-1.0

</details>

### Q3: Explain the difference between zero-shot, few-shot, and chain-of-thought prompting. When would you use each?

<details>
<summary>💡 Click to reveal answer</summary>

| Technique | How | When |
|-----------|-----|------|
| **Zero-shot** | Just ask the question | Simple factual queries, well-understood tasks |
| **Few-shot** | Provide input→output examples | Classification, formatting, pattern matching |
| **Chain-of-Thought** | "Think step by step" | Math, logic, multi-step reasoning |

**Key insight:** Few-shot is about *format*, CoT is about *reasoning*. You can combine them (few-shot CoT) for best results on complex tasks.

</details>

### Q4: What is the system prompt? How is it different from the user prompt? Can a user override it?

<details>
<summary>💡 Click to reveal answer</summary>

- **System prompt**: Sets persistent behavioral rules (persona, constraints, format). Set once.
- **User prompt**: The actual task/question. Changes each turn.
- System prompts have higher "authority" but are **not immune to prompt injection** — a malicious user can craft inputs that override system instructions.
- Defense: Input validation, output checking, guardrails, never put secrets in system prompts.

</details>

### Q5: How would you make an LLM application model-agnostic? Why would you want to?

<details>
<summary>💡 Click to reveal answer</summary>

**How:** Use an abstraction layer like `litellm` that normalizes different provider APIs (OpenAI, Gemini, Claude) into one common interface.

**Why:**
- **Avoid vendor lock-in** — switch providers without code changes
- **Cost optimization** — route simple tasks to cheaper models
- **Reliability** — fallback to another provider if one is down
- **A/B testing** — compare model quality across providers
- **Compliance** — some enterprises require specific providers for data residency

</details>

### Q6: What is "self-consistency" in prompting? When is it useful?

<details>
<summary>💡 Click to reveal answer</summary>

Self-consistency generates **multiple chain-of-thought reasoning paths** (using temperature > 0) and then takes the **majority vote** on the final answer.

**When useful:**
- High-stakes decisions where accuracy matters more than latency/cost
- Math problems where a single CoT might make an error
- Classification tasks where you want confidence scores

**Trade-off:** 3-5x more API calls (cost + latency) but significantly higher accuracy.

</details>

### Q7: You're building a customer support bot. What temperature would you set and why? What system prompt design would you use?

<details>
<summary>💡 Click to reveal answer</summary>

**Temperature: 0.2-0.3** — Low enough for consistency and accuracy, but not 0 (which can sound too robotic).

**System prompt design:**
```
You are a helpful customer support agent for [Company].
Rules:
- Be polite and empathetic
- Only answer questions about [Company] products
- If unsure, say "Let me connect you with a human agent"
- Never make up information about pricing or policies
- Keep responses under 3 sentences unless the user asks for details
```

Key principles: constrain scope, define fallback behavior, set output limits.

</details>

### Q8: What are the main differences between GPT-4, Gemini, and Claude? When would you choose each?

<details>
<summary>💡 Click to reveal answer</summary>

| Aspect | GPT-4o | Gemini 2.0 | Claude 3.5 |
|--------|--------|------------|------------|
| Strengths | Coding, function calling | Multi-modal, long context | Safety, nuanced writing |
| Context | 128K tokens | 1M+ tokens | 200K tokens |
| Best for | General purpose, tools | Document analysis, vision | Creative, careful reasoning |
| Cost | Medium | Lower | Medium |

**When to choose:**
- **GPT-4o**: Default choice, best tool/function calling support
- **Gemini**: Very long documents, multi-modal (images/video), cost-sensitive
- **Claude**: Tasks requiring nuance, safety-critical applications, long-form writing

</details>

---
## 🏗️ BUILD: Multi-Model Q&A Chatbot

Let's build a chatbot that:
1. Works with any LLM provider
2. Maintains conversation history
3. Supports system prompt customization
4. Shows token usage

In [ ]:
class MultiModelChatbot:
    """A production-ready chatbot that works with any LLM provider."""
    
    def __init__(self, model=None, system_prompt=None, temperature=0.7):
        self.model = model or DEFAULT_MODEL
        self.temperature = temperature
        self.system_prompt = system_prompt or "You are a helpful, friendly AI assistant. Keep responses concise."
        self.history = []
        self.total_tokens = 0
    
    def chat(self, user_message):
        """Send a message and get a response."""
        # Build message array
        messages = [{"role": "system", "content": self.system_prompt}]
        messages.extend(self.history)
        messages.append({"role": "user", "content": user_message})
        
        # Call LLM
        response = litellm.completion(
            model=self.model,
            messages=messages,
            temperature=self.temperature,
            max_tokens=500,
        )
        
        # Extract response
        assistant_message = response.choices[0].message.content
        
        # Track tokens
        usage = response.usage
        self.total_tokens += usage.total_tokens
        
        # Update history
        self.history.append({"role": "user", "content": user_message})
        self.history.append({"role": "assistant", "content": assistant_message})
        
        return {
            "response": assistant_message,
            "tokens_used": usage.total_tokens,
            "total_tokens": self.total_tokens,
            "model": self.model,
        }
    
    def reset(self):
        """Clear conversation history."""
        self.history = []
        self.total_tokens = 0
    
    def get_history(self):
        """Return formatted conversation history."""
        return self.history.copy()

# Create and test the chatbot
bot = MultiModelChatbot(
    system_prompt="You are a knowledgeable AI tutor specializing in computer science. Keep answers clear and under 3 sentences."
)

# Simulate a conversation
questions = [
    "What is a hash table?",
    "What's the average time complexity for lookup?",
    "When would you use a hash table vs a balanced BST?",
]

print(f"🤖 Chatbot using: {bot.model}")
print("=" * 60)

for q in questions:
    result = bot.chat(q)
    print(f"\n👤 You: {q}")
    print(f"🤖 Bot: {result['response']}")
    print(f"   [tokens: {result['tokens_used']} | total: {result['total_tokens']}]")

In [ ]:
# Interactive chat loop (run this cell to chat interactively)
# Uncomment the lines below to enable interactive mode

# bot = MultiModelChatbot(system_prompt="You are a helpful AI assistant.")
# print("🤖 Chat started! Type 'quit' to exit, 'reset' to clear history.")
# while True:
#     user_input = input("\n👤 You: ")
#     if user_input.lower() == 'quit':
#         print(f"\n👋 Goodbye! Total tokens used: {bot.total_tokens}")
#         break
#     if user_input.lower() == 'reset':
#         bot.reset()
#         print("🔄 History cleared!")
#         continue
#     result = bot.chat(user_input)
#     print(f"🤖 Bot: {result['response']}")

---
## ✅ Notebook 01 Summary

| Concept | Key Takeaway |
|---------|-------------|
| Multi-model | Use `litellm` for provider-agnostic code |
| Tokens | ~4 chars each, determine cost & context limits |
| Temperature | 0 = deterministic, 0.7 = balanced, 1.5 = creative |
| System prompts | Set persistent behavior rules, higher authority than user |
| Zero-shot | Just ask — good for simple tasks |
| Few-shot | Provide examples — good for formatting/classification |
| Chain-of-Thought | "Think step by step" — good for reasoning |
| Self-consistency | Multiple CoT + majority vote — good for accuracy |

### ➡️ Next: [Notebook 02 — Structured Output](./02_structured_output.ipynb)
*Learn how to force LLMs to output valid JSON, use function calling, and enforce schemas with Pydantic.*